In [0]:
from pyspark.sql import functions as F
from datetime import datetime
from decimal import Decimal

In [0]:
%run ./00_functions

In [0]:
# Create test data for transactions 
test_transactions = [
    # 3 valid rows
    (1003103, 2000075, 3000047, datetime(2024, 5, 4, 23, 44),
     'Pearly Pies', 28, Decimal('3.00'), Decimal('84.00')),

    (1000864, 2000290, 3000047, datetime(2024, 5, 6, 20, 46),
     'Golden Gate Ginger', 4, Decimal('3.00'), Decimal('12.00')),

    (1003222, 2000179, 3000046, datetime(2024, 5, 8, 10, 19),
     'Outback Oatmeal', 36, Decimal('3.00'), Decimal('108.00')),

    # Strict duplicate of the 1st row
    (1003103, 2000075, 3000047, datetime(2024, 5, 4, 23, 44),
     'Pearly Pies', 28, Decimal('3.00'), Decimal('84.00')),

    # dateTime NULL
    (1003230, 2000248, 3000046, None,
     'Tokyo Tidbits', 54, Decimal('3.00'), Decimal('162.00')),

    # product empty
    (1004001, 2000179, 3000046, datetime(2024, 5, 8, 10, 19),
     '', 10, Decimal('3.00'), Decimal('30.00')),

    # quantity = 0
    (1004002, 2000179, 3000046, datetime(2024, 5, 8, 11, 20),
     'Pearly Pies', 0, Decimal('3.00'), Decimal('0.00')),

    # Price mismatch
    (1004005, 2000179, 3000046, datetime(2024, 5, 8, 14, 20),
     'Tokyo Tidbits', 2, Decimal('3.00'), Decimal('10.00')),

    # Negative unitPrice
    (1004004, 2000179, 3000046, datetime(2024, 5, 8, 13, 20),
     'Outback Oatmeal', 2, Decimal('-3.00'), Decimal('-6.00')),

    # Missing transactionID
    (None, 2000179, 3000046, datetime(2024, 5, 8, 15, 20),
     'Pearly Pies', 2, Decimal('3.00'), Decimal('6.00')),

    # Unknown customer
    (1004006, 9999999, 3000046, datetime(2024, 5, 8, 16, 20),
     'Pearly Pies', 2, Decimal('3.00'), Decimal('6.00')),

    # Unknown franchise
    (1004007, 2000179, 9999999, datetime(2024, 5, 8, 17, 20),
     'Pearly Pies', 2, Decimal('3.00'), Decimal('6.00')),
]

columns = [
    'transactionID',
    'customerID',
    'franchiseID',
    'dateTime',
    'product',
    'quantity',
    'unitPrice',
    'totalPrice'
]

test_transactions_df = spark.createDataFrame(test_transactions, columns)
test_transactions_df.show(truncate=False)

In [0]:
# For customers 
test_customers = [
    # 2 Valid
    ("2001500", "Alice", "Martin", "Paris", "Ile-de-France",
     "France", "Europe", "female"),
    ("2001588", "John", "Smith", "Boston", "Massachusetts",
     "US", "North America", "male"),

    # Strict duplicate of C001
    ("2001500", "Alice", "Martin", "Paris", "Ile-de-France",
     "France", "Europe", "female"),

    #  Missing customerID
    (None, "Emma", "Brown", "London", "England",
     "UK", "Europe", "female"),

    # Missing first_name
    ("2001575", "", "Garcia", "Madrid", "Madrid",
     "Spain", "Europe", "female"),

    # Missing last_name
    ("2001512", "Lucas", None, "Lyon", "Auvergne-Rhone-Alpes",
     "France", "Europe", "male"),

    # Missing city
    ("2001589", "Anna", "Schmidt", "", "Bavaria",
     "Germany", "Europe", "female"),

    # Missing state
    ("2001523", "Kenji", "Tanaka", "Tokyo", None,
     "Japan", "Asia", "male"),

    # Missing country
    ("2001532", "Maria", "Silva", "Lisbon", "Lisbon",
     "", "Europe", "female"),

    # Invalid continent
    ("2001596", "Paul", "Wilson", "Chicago", "Illinois",
     "US", "Mars", "male"),

    # Invalid gender
    ("2001520", "Sophie", "Bernard", "Paris", "Ile-de-France",
     "France", "Europe", "unknown"),
]

customer_columns = [
    "customerID",
    "first_name",
    "last_name",
    "city",
    "state",
    "country",
    "continent",
    "gender"
]

test_customers_df = spark.createDataFrame(
    test_customers,
    customer_columns
)

In [0]:
# For franchises
test_franchises = [
    # 2 Valid
    ("3000111", "Golden Crumbs", "San Francisco", "Mission", "US", "XL"),
    ("3000121", "Tokyo Treats", "Tokyo", "Shibuya", "Japan", "M"),

    # Strict duplicate of F001
    ("3000111", "Golden Crumbs", "San Francisco", "Mission", "US", "XL"),

    # Missing franchiseID
    (None, "Sugar Rush", "Melbourne", "Fitzroy", "Australia", "L"),

    # Missing franchiseName
    ("3000123", "", "Sydney", "Bondi Beach", "Australia", "L"),

    # Missing city
    ("3000189", "Sweet Dreams", "", "Wicker Park", "US", "XXL"),

    # Missing district
    ("3000145", "Kyoto Kravings", "Kyoto", None, "Japan", "S"),

    # Missing country
    ("3000178", "The Baking Lab", "Perth", "Leederville", "", "L"),

    # Invalid size
    ("3000695", "Crumbly Delights", "Brisbane", "West End",
     "Australia", "HUGE"),
]

franchise_columns = [
    "franchiseID",
    "franchiseName",
    "city",
    "district",
    "country",
    "size"
]

test_franchises_df = spark.createDataFrame(
    test_franchises,
    franchise_columns
)

In [0]:
# Standarization 
test_std_transactions = standarize_transactions(test_transactions_df)
test_std_customers = standarize_customers(test_customers_df)
test_std_franchises = standarize_franchises(test_franchises_df)

In [0]:
# Deduplication
test_dedup_transactions = check_global_quality(test_std_transactions, 'transactionID')
test_dedup_customers = check_global_quality(test_std_customers, 'customerID')
test_dedup_franchises = check_global_quality(test_std_franchises, 'franchiseID')

In [0]:
# Validation data 
test_checked_customers = validate_customers(test_dedup_customers)
test_checked_franchises = validate_franchises(test_dedup_franchises)

checked_transactions_fact = validate_transactions(test_dedup_transactions, test_checked_customers, test_checked_franchises)